# Linear Programming in SciPy

In [ ]:
import numpy as np
from scipy.optimize import linprog

# x = [x_aw, x_af, x_bw, x_bf]

# linprog minimizes c @ x.
# Negate the coefficients to maximize the original objective.
c = -np.array([2, 1, 1, 3], dtype=float)

In [ ]:
# Define constraints
A_ub = np.array(
    [
        [1, 1, 0, 0],  # User A's time
        [0, 0, 1, 1],  # User B's time
    ],
    dtype=float,
)

b_ub = np.array([10, 8], dtype=float)

In [ ]:
# Solve the linear program
bounds = (0, None)

result = linprog(
    c,
    A_ub=A_ub,
    b_ub=b_ub,
    bounds=bounds,
    method="highs",
)

In [4]:
if not result.success:
    raise RuntimeError(result.message)

print(result.message)
# Optimization terminated successfully. (HiGHS Status 7: Optimal)

Optimization terminated successfully. (HiGHS Status 7: Optimal)


In [ ]:
x_aw, x_af, x_bw, x_bf = result.x

print(f"x_aw = {x_aw:.2f}")
print(f"x_af = {x_af:.2f}")
print(f"x_bw = {x_bw:.2f}")
print(f"x_bf = {x_bf:.2f}")
print(f"Maximum content score = {-result.fun:.2f}")
# x_aw = 10.00
# x_af = 0.00
# x_bw = 0.00
# x_bf = 8.00
# Maximum content score = 44.00

x_aw = 10.00
x_af = 0.00
x_bw = 0.00
x_bf = 8.00
Maximum content score = 44.00


# Integer Programming

In [ ]:
import numpy as np
from scipy.optimize import Bounds, LinearConstraint, milp
from itertools import product


users = ["u0", "u1"]
products = ["p0", "p1", "p2"]

pairs = list(product(users, products))

costs = {(u, p): 4 if p == "p0" else 6 for u, p in pairs}

revenues = {(u, p): 6 if u == "u0" else 4 for u, p in pairs}

budget = 20
K = 2
q = {p: 1 for p in products}

In [8]:
# Objective
c = -np.array(
    [revenues[pair] for pair in pairs],
    dtype=float,
)

In [ ]:
A = []
lower = []
upper = []

# 1. Total expected cost must not exceed the budget.
A.append([costs[pair] for pair in pairs])
lower.append(-np.inf)
upper.append(budget)

# 2. Each user receives exactly K recommendations.
for u in users:
    A.append([1 if pair[0] == u else 0 for pair in pairs])
    lower.append(K)
    upper.append(K)

# 3. Each program must meet its minimum expected-revenue threshold.
for p in products:
    A.append([revenues[pair] if pair[1] == p else 0 for pair in pairs])
    lower.append(q[p])
    upper.append(np.inf)

constraints = LinearConstraint(
    np.array(A, dtype=float),
    np.array(lower, dtype=float),
    np.array(upper, dtype=float),
)

In [10]:
bounds = Bounds(
    np.zeros(len(pairs)),
    np.ones(len(pairs)),
)

integrality = np.ones(len(pairs), dtype=int)

In [11]:
result = milp(
    c,
    integrality=integrality,
    bounds=bounds,
    constraints=constraints,
)

if not result.success:
    raise RuntimeError(result.message)

In [ ]:
solution = np.rint(result.x).astype(int)

for pair, selected in zip(pairs, solution):
    print(f"{pair}: {selected}")

print(f"Maximum expected revenue = {-result.fun:.2f}")
# ('u0', 'p0'): 1
# ('u0', 'p1'): 1
# ('u0', 'p2'): 0
# ('u1', 'p0'): 1
# ('u1', 'p1'): 0
# ('u1', 'p2'): 1
# Maximum expected revenue = 20.00

('u0', 'p0'): 1
('u0', 'p1'): 1
('u0', 'p2'): 0
('u1', 'p0'): 1
('u1', 'p1'): 0
('u1', 'p2'): 1
Maximum expected revenue = 20.00
